# UrbanCart - Part 1: Exploratory & Business Analysis
Data & AI Engineer case study. All logic lives in `src/` (reusable modules); this notebook calls it and shows the results.

**Definitions (from the brief, unchanged)**
- *Successful order* = status `delivered`. *New customer* = user with >= 1 successful order, counted under their acquisition channel.
- CAC = channel spend / new customers. Revenue per customer = delivered revenue / customers (Jan-Jun). Repeat rate = customers with >= 2 delivered orders / customers.
- LTV = revenue per customer x 30%. LTV:CAC = LTV / CAC.
- **Customer denominator everywhere = customers with at least one delivered order** (not signups).
- Money is rounded to the nearest rupee, percentages to 1 decimal (LTV:CAC to 2 decimals).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))          # project root, so `src` is importable
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 90)

from src import config as C
from src.automation.pipeline import run_data_preparation, run_analysis

## 1. Data quality (raw files are never modified)

In [2]:
quality, cleaned, exclusions, reconciliation = run_data_preparation()
print(f"{len(quality)} checks run  ->  {(quality.status=='ISSUE').sum()} issues, {(quality.status=='INFO').sum()} informational, {(quality.status=='OK').sum()} passed")
quality[quality.status.isin(['ISSUE','INFO'])][['issue_id','status','table','column','check','rows_affected','pct_affected']]

43 checks run  ->  4 issues, 6 informational, 33 passed


,issue_id,status,table,column,check,rows_affected,pct_affected
3,DQ04,ISSUE,users,acquisition_channel,Missing values in acquisition_channel,72,1.4
13,DQ14,ISSUE,users,email,Internal QA/test accounts (@urbancart.in),35,0.7
14,DQ15,INFO,users,user_id,Users with no orders at all,2097,40.8
20,DQ21,ISSUE,orders,order_id,Duplicate order_id,206,3.6
21,DQ22,INFO,orders,all,Exact duplicate order rows,206,3.6
26,DQ27,INFO,orders,status,Non-delivered orders (cancelled + returned),735,12.9
31,DQ32,INFO,orders,order_date,Orders placed on the signup day,320,5.6
32,DQ33,ISSUE,orders,user_id,Orders from internal QA/test accounts,310,5.5
33,DQ34,INFO,orders,order_value,Statistical outliers in order_value (IQR rule),44,0.8
42,DQ43,INFO,spend,spend_inr,Total spend sanity vs brief ('about Rs 16 lakh'),0,0.0


In [3]:
# What was found, how it was handled and why (issues only)
for _, r in quality[quality.status=='ISSUE'].iterrows():
    print(f"[{r.issue_id}] {r.check}  |  {r.rows_affected} rows ({r.pct_affected}%)")
    print(f"  Finding : {r.issue_description}")
    print(f"  Handling: {r.handling}")
    print(f"  Why     : {r.business_justification}")
    print(f"  Impact  : {r.impact_if_ignored}")
    print()

[DQ04] Missing values in acquisition_channel  |  72 rows (1.4%)
  Finding : 72 users have a blank acquisition channel (40 of them have at least one delivered order).
  Handling: Retained and labelled 'Unknown'. NOT mapped to Organic and NOT imputed. Excluded from per-channel metrics, reported separately.
  Why     : The true source is unknowable; guessing (e.g. Organic) would silently move customers and revenue between channels and distort CAC / LTV.
  Impact  : 40 real customers would be attributed to the wrong channel (or none).

[DQ14] Internal QA/test accounts (@urbancart.in)  |  35 rows (0.7%)
  Finding : 35 users have the company's own email domain (35 follow the 'qa.test<N>' pattern). All carry channel 'Affiliate' (1 distinct channel) and city 'Delhi' (1 distinct city); 35 of them have delivered orders (285 unique orders, Rs 779,790).
  Handling: Excluded from all metrics together with their orders; logged in excluded_records.csv (not deleted from raw).
  Why     : Test accounts

In [4]:
# Nothing is silently dropped: raw = cleaned + excluded
reconciliation

,table,raw_rows,excluded_rows,cleaned_rows
0,users,5135,35,5100
1,orders,5684,491,5193
2,spend,24,0,24


In [5]:
exclusions.groupby(['source_table','reason']).size().rename('rows').reset_index()

,source_table,reason,rows
0,orders,exact_duplicate_row,206
1,orders,internal_qa_test_account_order,285
2,users,internal_qa_test_account,35


**Checks that passed (no problem found):** duplicate user_ids/emails, invalid or out-of-range dates, invalid channel names and statuses, missing/zero/negative order values, orders for non-existent users, orders before signup, spend completeness (4 channels x 6 months, total Rs 15,90,000 as in the brief).

## 2. Run the analysis

In [6]:
res = run_analysis(cleaned)
customers = res["customers"]
print(f"{len(customers):,} customers (users with >=1 delivered order); invariant checks passed: {len(res['invariants_passed'])}")

2,643 customers (users with >=1 delivered order); invariant checks passed: 12


## 2.1 Acquisition cost (paid channels, Jan-Jun 2026)
CAC = total channel spend / new customers. Signups are context only.

In [7]:
res['acquisition']

,channel,total_spend,signups,new_customers,cac,signup_to_customer_pct
0,Google Ads,400000,1100,655,611.0,59.5
1,Meta Ads,547000,1300,675,810.0,51.9
2,Affiliate,346000,1200,505,685.0,42.1
3,Influencer,297000,600,307,967.0,51.2
4,All paid,1590000,4200,2142,742.0,51.0


## 2.2 Customer value (all channels incl. Organic)
Denominator: customers with >= 1 delivered order. `Unknown` = 40 customers whose channel was blank in the source (not mapped to Organic).

In [8]:
res['customer_value']

,channel,customers,total_revenue,repeat_customers,revenue_per_customer,repeat_purchase_rate_pct,customer_denominator
0,Google Ads,655,1513880,378,2311,57.7,customers with >=1 delivered order
1,Meta Ads,675,1124892,302,1667,44.7,customers with >=1 delivered order
2,Affiliate,505,483387,41,957,8.1,customers with >=1 delivered order
3,Influencer,307,877469,224,2858,73.0,customers with >=1 delivered order
4,Organic,461,877167,257,1903,55.7,customers with >=1 delivered order
5,Unknown,40,85089,21,2127,52.5,customers with >=1 delivered order


## 2.3 LTV and LTV:CAC
LTV = revenue per customer x 30%. **Limitation:** revenue covers only Jan-Jun 2026 (0-6 months of customer life), so this is a *realised* value, not a lifetime forecast; no extrapolation is applied. Organic has no spend, so no CAC / LTV:CAC is calculated for it.

In [9]:
res['ltv']

,channel,revenue_per_customer,contribution_margin_pct,ltv,cac,ltv_cac_ratio,net_value_per_customer,repeat_purchase_rate_pct,ltv_cac_rank,note
0,Google Ads,2311.0,30.0,693.0,611.0,1.14,83.0,57.7,1,LTV above CAC
1,Meta Ads,1667.0,30.0,500.0,810.0,0.62,-310.0,44.7,3,LTV below CAC (not recovered by Jan-Jun margin)
2,Affiliate,957.0,30.0,287.0,685.0,0.42,-398.0,8.1,4,LTV below CAC (not recovered by Jan-Jun margin)
3,Influencer,2858.0,30.0,857.0,967.0,0.89,-110.0,73.0,2,LTV below CAC (not recovered by Jan-Jun margin)
4,Organic,1903.0,30.0,571.0,NaN,NaN,NaN,55.7,<NA>,No organic marketing spend supplied -> CAC and LTV:CAC not calculated


## 2.4 Monthly CAC trend (customers grouped by signup month)

In [10]:
m = res['monthly']
print("CAC (Rs)"); display(m.pivot(index='channel', columns='month', values='cac').loc[C.PAID_CHANNELS])
print("New customers (signup month, >=1 delivered order)"); display(m.pivot(index='channel', columns='month', values='new_customers').loc[C.PAID_CHANNELS])
print("Spend (Rs)"); display(m.pivot(index='channel', columns='month', values='spend').loc[C.PAID_CHANNELS])

CAC (Rs)


month,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
channel,,,,,,
Google Ads,592.0,621.0,596.0,602.0,644.0,611.0
Meta Ads,491.0,618.0,789.0,886.0,958.0,1115.0
Affiliate,709.0,709.0,728.0,633.0,667.0,682.0
Influencer,981.0,980.0,913.0,960.0,964.0,1000.0


New customers (signup month, >=1 delivered order)


month,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
channel,,,,,,
Google Ads,125,95,104,123,118,90
Meta Ads,110,110,109,123,119,104
Affiliate,86,79,81,109,84,66
Influencer,52,51,46,50,56,52


Spend (Rs)


month,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
channel,,,,,,
Google Ads,74000,59000,62000,74000,76000,55000
Meta Ads,54000,68000,86000,109000,114000,116000
Affiliate,61000,56000,59000,69000,56000,45000
Influencer,51000,50000,42000,48000,54000,52000


In [11]:
res['change']

,channel,jan_cac,jun_cac,abs_change,pct_change,slope_per_month,direction_jan_to_jun,months_with_undefined_cac
0,Google Ads,592.0,611.0,19.0,3.2,5.0,increasing,0
1,Meta Ads,491.0,1115.0,624.0,127.2,121.0,increasing,0
2,Affiliate,709.0,682.0,-27.0,-3.9,-10.0,decreasing,0
3,Influencer,981.0,1000.0,19.0,2.0,3.0,increasing,0


No month has zero new customers, so no CAC is undefined here; the code returns NaN (shown as 'undefined') rather than dividing by zero if that ever happens.

## 2.5 Sanity check

In [12]:
cmp = res['comparison']
print(f"Independent recalculation from RAW csv (stdlib only, no pandas/DuckDB): {cmp.match.sum()}/{len(cmp)} metrics match")
print(Path('../outputs/part1_analysis/sanity_check.md').read_text(encoding='utf-8')[:1800])

Independent recalculation from RAW csv (stdlib only, no pandas/DuckDB): 60/60 metrics match
# Sanity Check (Part 1, question 6)

## Number selected: Meta Ads CAC

- **Spend (read directly from raw `marketing_spend.csv`)**: 54,000 + 68,000 + 86,000 + 109,000 + 114,000 + 116,000 = **Rs 547,000**
- **New customers (independent method)**: **675** users tagged Meta Ads with >= 1 delivered order
- **CAC** = 547,000 / 675 = **Rs 810**
- **Main pipeline result**: Rs 810  ->  **agrees**

## How the second method differs from the first

| | Main pipeline | Independent check |
|---|---|---|
| Tools | pandas + DuckDB SQL | Python standard-library `csv`, plain dicts and sets |
| Input | Cleaned CSVs in `data/processed/` | **Raw** CSVs in `data/raw/` |
| Dedup + test-account removal | `clean_data.py` | Re-implemented separately (set of full rows; email domain) |
| Aggregation | SQL JOIN / GROUP BY | Loops over rows, per-user counters |
| Spend | Sum of cleaned spend table | Sum of the raw spend colu

## 3. Key observations (all numbers from the tables above)
- **CAC:** Google Ads Rs 611 < Affiliate Rs 685 < Meta Ads Rs 810 < Influencer Rs 967.
- **Best / worst LTV:CAC:** Google Ads 1.14 (best) ... Affiliate 0.42 (worst). Only Google Ads has LTV above CAC on the Jan-Jun margin.
- **Influencer** has the highest CAC but also the highest revenue per customer (Rs 2,858) and repeat rate (73.0%) - LTV:CAC 0.89, well above Meta and Affiliate. It is the clearest case where cutting on CAC alone would mislead.
- **Affiliate:** lowest revenue per customer (Rs 957) and a repeat rate of only 8.1%, and it converts the fewest signups (42.1%).
- **Meta Ads:** CAC rose from Rs 491 (Jan) to Rs 1,115 (Jun), +127.2%, while its spend more than doubled (Rs 54k -> Rs 116k) and new customers stayed flat (110 -> 104). The other channels moved by less than +/-4%.
- **Caveats:** one six-month window; ~50-125 new customers per channel-month; association, not causation; signup-month cohorts that signed up late have had less time to place a first delivered order.